<a href="https://colab.research.google.com/github/ipeirotis-org/datasets/blob/main/Foot_Traffic/Dewey_Weekly_Patterns_NY_Restaurants.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Foot Traffic: Weekly Patterns Plus (NY Restaurants)

This notebook downloads the **Weekly Patterns Plus** dataset (customized for NY Restaurants) from Dewey Data, converts it to Parquet, uploads to Google Cloud Storage, and creates a BigQuery external table.

**Dataset details:**
- Source: Dewey Data (SafeGraph / Advan)
- Rows: ~134.6M
- Columns: 38
- Size: ~82 GB (compressed CSV)
- Scope: New York restaurants, weekly foot traffic patterns

**References:**
- [Dewey Data Client Docs](https://docs.deweydata.io/docs/dewey-client)
- [Weekly Patterns Plus Schema](https://docs.deweydata.io/docs/advan-wp)

## Setup and Authentication

In [ ]:
!pip install -q deweypy google-cloud-bigquery google-cloud-storage pyarrow pandas

In [ ]:
from google.colab import auth

auth.authenticate_user()

In [ ]:
import os
import glob
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.csv as pcsv
from google.cloud import storage
from google.cloud import bigquery

In [ ]:
# Dewey API key — store in Colab Secrets (key name: DEWEY_API_KEY)
# See: https://x.com/GoogleColab/status/1719798901042991166
from google.colab import userdata

DEWEY_API_KEY = userdata.get("DEWEY_API_KEY")
DEWEY_PROJECT_ID = "prj_zyknaorh__cdst_36a9hebgkfo4foxj"

PROJECT_ID = "nyu-datasets"
GCS_BUCKET_NAME = "dewey-foot-traffic"
GCS_FOLDER = "weekly_patterns_ny_restaurants"
BQ_DATASET_ID = "foot_traffic"
BQ_TABLE_ID = "weekly_patterns_ny_restaurants"

DOWNLOAD_DIR = "dewey_downloads"
PARQUET_DIR = "parquet_output"

os.makedirs(DOWNLOAD_DIR, exist_ok=True)
os.makedirs(PARQUET_DIR, exist_ok=True)

In [ ]:
storage_client = storage.Client(project=PROJECT_ID)
bucket = storage_client.bucket(GCS_BUCKET_NAME)
bigquery_client = bigquery.Client(project=PROJECT_ID)

## 1. Download Data from Dewey

Uses the `deweypy` CLI for multi-threaded download. Files arrive as `.csv.gz`.

The dataset is ~82 GB, so this will take a while. You can use the `--partition-key-after` and `--partition-key-before` flags to download date ranges incrementally.

In [ ]:
# Download all files (this may take several hours for the full 82 GB dataset)
# Add --partition-key-after YYYY-MM-DD --partition-key-before YYYY-MM-DD to filter by date
!python -m deweypy --api-key {DEWEY_API_KEY} speedy-download {DEWEY_PROJECT_ID} --destination {DOWNLOAD_DIR}

In [ ]:
# List downloaded files
downloaded_files = sorted(glob.glob(f"{DOWNLOAD_DIR}/**/*.csv.gz", recursive=True))
if not downloaded_files:
    downloaded_files = sorted(glob.glob(f"{DOWNLOAD_DIR}/**/*.csv", recursive=True))

total_size_gb = sum(os.path.getsize(f) for f in downloaded_files) / (1024**3)
print(f"Downloaded {len(downloaded_files)} files, total size: {total_size_gb:.2f} GB")
for f in downloaded_files[:5]:
    print(f"  {f} ({os.path.getsize(f) / (1024**2):.1f} MB)")
if len(downloaded_files) > 5:
    print(f"  ... and {len(downloaded_files) - 5} more files")

## 2. Inspect the Data

In [ ]:
# Read a sample to understand the schema
sample_file = downloaded_files[0]
sample_df = pd.read_csv(sample_file, nrows=100, dtype="str")
print(f"Columns ({len(sample_df.columns)}):")
for i, col in enumerate(sample_df.columns):
    print(f"  {i+1:2d}. {col}")

In [ ]:
sample_df.head()

In [ ]:
sample_df.dtypes

## 3. Convert to Parquet and Upload to GCS

Process each CSV.GZ file: read with PyArrow (memory-efficient), write as Parquet, upload to GCS. This avoids loading the entire 82 GB into memory.

In [ ]:
def upload_to_gcs(local_file, destination_blob_name):
    blob = bucket.blob(destination_blob_name)
    blob.upload_from_filename(local_file)
    print(f"  Uploaded to gs://{GCS_BUCKET_NAME}/{destination_blob_name}")

In [ ]:
for i, csv_file in enumerate(downloaded_files):
    base_name = os.path.splitext(os.path.splitext(os.path.basename(csv_file))[0])[0]
    parquet_file = os.path.join(PARQUET_DIR, f"{base_name}.parquet")
    gcs_destination = f"{GCS_FOLDER}/parquet/{base_name}.parquet"

    # Check if already uploaded
    blob = bucket.blob(gcs_destination)
    if blob.exists():
        print(f"[{i+1}/{len(downloaded_files)}] {base_name} — already in GCS, skipping")
        continue

    print(f"[{i+1}/{len(downloaded_files)}] Converting {base_name}...")

    # Read CSV with PyArrow (handles .csv.gz automatically)
    table = pcsv.read_csv(csv_file)

    # Write as Parquet with snappy compression
    pq.write_table(table, parquet_file, compression='snappy')

    # Upload to GCS
    upload_to_gcs(parquet_file, gcs_destination)

    # Clean up local parquet file to save disk space
    os.remove(parquet_file)

print(f"\nDone! Processed {len(downloaded_files)} files.")

## 4. Verify GCS Upload

In [ ]:
blobs = list(bucket.list_blobs(prefix=f"{GCS_FOLDER}/parquet/"))
total_gcs_size_gb = sum(b.size for b in blobs) / (1024**3)
print(f"GCS files: {len(blobs)}")
print(f"Total size in GCS: {total_gcs_size_gb:.2f} GB")
for b in blobs[:5]:
    print(f"  {b.name} ({b.size / (1024**2):.1f} MB)")
if len(blobs) > 5:
    print(f"  ... and {len(blobs) - 5} more files")

## 5. Create BigQuery External Table

Creates a BigQuery external table pointing at the Parquet files in GCS. BigQuery reads the schema directly from the Parquet metadata — no need to define it manually.

In [ ]:
# Create the dataset if it doesn't exist
dataset_ref = f"{PROJECT_ID}.{BQ_DATASET_ID}"
try:
    bigquery_client.get_dataset(BQ_DATASET_ID)
    print(f"Dataset {BQ_DATASET_ID} already exists.")
except Exception:
    dataset = bigquery.Dataset(dataset_ref)
    dataset.location = "US"
    dataset = bigquery_client.create_dataset(dataset, exists_ok=True)
    print(f"Created dataset {BQ_DATASET_ID}")

In [ ]:
# Create external table on GCS Parquet files
table_ref = bigquery_client.dataset(BQ_DATASET_ID).table(BQ_TABLE_ID)

external_config = bigquery.ExternalConfig("PARQUET")
external_config.source_uris = [f"gs://{GCS_BUCKET_NAME}/{GCS_FOLDER}/parquet/*.parquet"]
external_config.autodetect = True

table = bigquery.Table(table_ref)
table.external_data_configuration = external_config

# Delete existing table if it exists, then create
bigquery_client.delete_table(table_ref, not_found_ok=True)
table = bigquery_client.create_table(table)
print(f"Created external table {PROJECT_ID}.{BQ_DATASET_ID}.{BQ_TABLE_ID}")

## 6. Validate

In [ ]:
# Check row count
query = f"SELECT COUNT(*) as row_count FROM `{PROJECT_ID}.{BQ_DATASET_ID}.{BQ_TABLE_ID}`"
result = bigquery_client.query(query).to_dataframe()
print(f"Total rows in BigQuery: {result['row_count'][0]:,}")

In [ ]:
# Preview the schema
table_info = bigquery_client.get_table(f"{PROJECT_ID}.{BQ_DATASET_ID}.{BQ_TABLE_ID}")
print(f"Table: {table_info.full_table_id}")
print(f"Rows: {table_info.num_rows:,}" if table_info.num_rows else "Rows: (external table)")
print(f"\nSchema ({len(table_info.schema)} columns):")
for field in table_info.schema:
    print(f"  {field.name:40s} {field.field_type:15s} {field.mode}")

In [ ]:
# Sample a few rows
query = f"SELECT * FROM `{PROJECT_ID}.{BQ_DATASET_ID}.{BQ_TABLE_ID}` LIMIT 5"
sample = bigquery_client.query(query).to_dataframe()
sample

## 7. Add Table Description

In [ ]:
alter_sql = f"""
ALTER TABLE `{PROJECT_ID}.{BQ_DATASET_ID}.{BQ_TABLE_ID}`
SET OPTIONS (
  description = 'Weekly foot traffic patterns for NY restaurants. Source: Dewey Data (Advan/SafeGraph Weekly Patterns Plus). ~134M rows covering weekly visit patterns including visitor counts, dwell times, visitor home locations, and popularity metrics.'
);
"""
bigquery_client.query(alter_sql).result()
print("Table description updated.")

## 8. Sample Queries

In [ ]:
sample_queries = f"""
-- Total visits by week
SELECT
  date_range_start,
  SUM(raw_visit_counts) AS total_visits,
  SUM(raw_visitor_counts) AS total_visitors,
  COUNT(DISTINCT placekey) AS num_restaurants
FROM `{PROJECT_ID}.{BQ_DATASET_ID}.{BQ_TABLE_ID}`
GROUP BY date_range_start
ORDER BY date_range_start;

-- Top restaurants by total visits
SELECT
  placekey,
  location_name,
  street_address,
  city,
  postal_code,
  SUM(raw_visit_counts) AS total_visits
FROM `{PROJECT_ID}.{BQ_DATASET_ID}.{BQ_TABLE_ID}`
GROUP BY placekey, location_name, street_address, city, postal_code
ORDER BY total_visits DESC
LIMIT 50;

-- Average dwell time and visits per restaurant per week
SELECT
  date_range_start,
  AVG(raw_visit_counts) AS avg_visits,
  AVG(median_dwell) AS avg_median_dwell_minutes
FROM `{PROJECT_ID}.{BQ_DATASET_ID}.{BQ_TABLE_ID}`
GROUP BY date_range_start
ORDER BY date_range_start;
"""
print(sample_queries)

## 9. Cleanup (Optional)

Remove local downloaded files to free disk space.

In [ ]:
# Uncomment to remove local files after successful upload
# import shutil
# shutil.rmtree(DOWNLOAD_DIR)
# shutil.rmtree(PARQUET_DIR, ignore_errors=True)
# print("Local files removed.")